In [4]:
# Install required libraries
!pip install transformers pandas torch qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 19.5 MB/s eta 0:00:00


In [5]:
import pandas as pd
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from torch.amp import autocast
import os

In [6]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clear the CUDA cache to free up memory
torch.cuda.empty_cache()

MessageError: Error: credential propagation was unsuccessful

In [7]:
# Clear the CUDA cache to free up memory
torch.cuda.empty_cache()

# Load the model on the available device(s)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load the default processor for the model
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

In [13]:
import requests
import pandas as pd
import torch
from torch.amp import autocast  # ✅ Updated import

# GitHub repository URL for raw file access
github_repo_url = "https://raw.githubusercontent.com/Namprire/multimodal/test1/"

# Download the CSV file containing the questions and options
csv_url = github_repo_url + "Validation/validation_without_answers.csv"
csv_file = "validation_without_answers.csv"

response = requests.get(csv_url)
if response.status_code == 200:
    with open(csv_file, "wb") as file:
        file.write(response.content)
else:
    raise Exception(f"Failed to download CSV. Status code: {response.status_code}")

# Read the CSV file
df = pd.read_csv(csv_file)

# Get the list of image files from the GitHub repository
image_folder_url = github_repo_url + "Validation/images/"
image_files = df["file_name"].tolist()

# Prepare the output CSV file
output_csv_file = "test_validation.csv"
output_data = []

# Loop through each image file in the validation folder
for image_file in image_files:
    image_url = image_folder_url + image_file
    image_path = image_file

    # Download the image file
    response = requests.get(image_url)
    if response.status_code == 200:
        with open(image_path, "wb") as file:
            file.write(response.content)
    else:
        print(f"Failed to download {image_file}. Skipping...")
        continue

    # Get the corresponding row from the CSV file based on the image file name
    row = df[df["file_name"] == image_file].iloc[0]
    question = row["question"]
    options = [row["option1"], row["option2"], row["option3"], row["option4"]]

    # Prepare the options text for the prompt
    options_text = "\n".join([f"{chr(97 + i)}) {option}" for i, option in enumerate(options)])
    prompt = f"Question: {question}\nOptions:\n{options_text}\nPlease choose the correct option (a, b, c, or d) and explain why you chose this answer:"

    # Prepare the message for the model, including the image and the prompt
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # Apply the chat template to the messages and process the vision information
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    # Prepare the inputs for the model
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")

    # ✅ Fix: Use `torch.amp.autocast` instead of `torch.cuda.amp.autocast`
    with torch.amp.autocast(device_type="cuda"):
        generated_ids = model.generate(**inputs, max_new_tokens=128)

    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    # Find the model's answer among the options
    model_answer = None
    for i, option in enumerate(options, 1):
        if option in output_text[0]:
            model_answer = i
            break

    # Append the result to the output data
    output_data.append([image_file, model_answer])

    # Free GPU memory
    del inputs
    del generated_ids
    del generated_ids_trimmed
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Write the output data to the CSV file
output_df = pd.DataFrame(output_data, columns=["file_name", "answer"])
output_df.to_csv(output_csv_file, index=False)

print(f"Validation results saved to {output_csv_file}")


OutOfMemoryError: CUDA out of memory. Tried to allocate 84.20 GiB. GPU 0 has a total capacity of 14.75 GiB of which 3.21 GiB is free. Process 4321 has 11.54 GiB memory in use. Of the allocated memory 9.97 GiB is allocated by PyTorch, and 1.44 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)